# A1 — Priprava podatkov: odhod naročnikov

Ponovimo znane korake prvega dne in pripravimo eno vrstico na naročnika.
Cilj je `odhod` (0/1).

**Rezultat:** `data/pripravljeno/telekom_modeliranje.csv` in spremljajoča shema JSON
z opisi, vlogami ter nabori značilk.

Izvoz nima potrjenega časovnega horizonta odhoda ali datumov posameznih dogodkov.
Ne ustvarjamo izmišljenih »zadnjih 30 dni«. Za učno primerjavo predpostavimo, da
izbrani agregati opisujejo stanje pred izidom; za dejansko uporabo bi to morali preveriti.

**Odločitev:** zdaj izvajamo pravila, ki ne zahtevajo učenja iz cele zbirke: čiščenje,
smiselne agregate, razmerja, fiksne časovne oznake in kazalnike manjkanja.
Neznane številke ostanejo `NaN`, neznane kategorije dobijo `NEZNANO`.

Čista tabela še ni končna numerična matrika za model. Mediano, skaliranje,
one-hot kodiranje, združevanje redkih kategorij in izbor značilk bomo določili
**po delitvi**, samo na učnih podatkih. Tukaj ničesar ne delimo in ne učimo modelov.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

KOREN = Path.cwd().resolve()
while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
    KOREN = KOREN.parent
if not (KOREN / "utils/orodja.py").is_file():
    raise FileNotFoundError("Odprite zvezek znotraj mape projekta.")
if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
from utils.orodja import poskrbi_za_podatke

DATA = poskrbi_za_podatke()
IZHOD = DATA / "pripravljeno"
IZHOD.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 15)
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11})

## 1. Naložimo in preverimo ključe
Odstranimo samo povsem enake vrstice. Dva različna zapisa iste stranke zahtevata
razrešitev, zato tak primer ustavi pripravo. Neznanega cilja ne dopolnjujemo.

In [ ]:
surovi = pd.read_csv(DATA / "telekom_izvirni.csv", dtype={"Customer_ID": "string"})
print("Vhod:", surovi.shape, "povsem podvojenih vrstic:", surovi.duplicated().sum())
surovi = surovi.drop_duplicates().copy()
surovi["Customer_ID"] = surovi["Customer_ID"].str.strip().replace("", pd.NA)
assert not surovi.loc[surovi["Customer_ID"].notna(), "Customer_ID"].duplicated().any(), "Nasprotujoči zapisi iste stranke."
surovi["churn"] = pd.to_numeric(surovi["churn"], errors="coerce")
veljavno = surovi["Customer_ID"].notna() & surovi["churn"].isin([0, 1])
print("Izločene vrstice brez veljavnega ID ali cilja:", int((~veljavno).sum()))
surovi = surovi.loc[veljavno].copy()
display(surovi["churn"].value_counts().sort_index().rename("naročniki").to_frame())

## 2. Izberemo širši, razumljiv nabor
Ohranimo znanih 20 vhodnih stolpcev iz prvega dne in dodamo merila porabe,
kakovosti klicev, naprave ter 3- in 6-mesečne agregate. Vseh 100 stolpcev ne
vključimo avtomatično: nejasno dokumentirane kode pustimo zunaj.
Demografske podatke ločimo v svojo skupino, da jih lahko pri primerjavi izpustimo.

In [ ]:
from utils.orodja import TELEKOM_STOLPCI
osnovni_slovar = {ime: (vir, opis) for ime, (vir, opis) in TELEKOM_STOLPCI.items()
                  if ime not in ["id_stranke", "odhod"]}
# Enota spremembe v viru ni dovolj potrjena za razlago kot odstotek.
osnovni_slovar["sprememba_minut"] = ("change_mou", "Sprememba minut; ohranimo izvorno mero, ne trdimo, da je odstotek.")
osnovni_slovar["povp_gostovanje"] = ("roam_Mean", "Izvorni povprečni agregat gostovanja; enoto preverimo v podatkovnem viru.")
dodatni_slovar = {
    "povp_minute_3m": ("avg3mou", "Povprečne mesečne minute v 3-mesečnem agregatu."),
    "povp_prihodek_3m": ("avg3rev", "Povprečni mesečni prihodek v 3-mesečnem agregatu, USD."),
    "povp_prihodek_6m": ("avg6rev", "Povprečni mesečni prihodek v 6-mesečnem agregatu, USD."),
    "povp_klici_3m": ("avg3qty", "Povprečno število klicev v 3-mesečnem agregatu."),
    "povp_klici_6m": ("avg6qty", "Povprečno število klicev v 6-mesečnem agregatu."),
    "sprememba_prihodka": ("change_rev", "Izvorna sprememba prihodka; lahko je negativna."),
    "aktivne_narocnine": ("actvsubs", "Število aktivnih naročnin."),
    "stevilo_modelov_telefonov": ("models", "Število modelov telefonov, ki jih je stranka imela."),
    "obnovljena_naprava": ("refurb_new", "Izvorna koda nove oziroma obnovljene naprave (N/R)."),
    "povp_blokirani_klici": ("blck_vce_Mean", "Povprečno število blokiranih glasovnih klicev."),
    "povp_odhodni_klici": ("plcd_vce_Mean", "Povprečno število poskusov odhodnih glasovnih klicev."),
    "povp_dohodni_klici": ("recv_vce_Mean", "Povprečno število prejetih glasovnih klicev."),
    "povp_uspesni_klici": ("comp_vce_Mean", "Povprečno število vzpostavljenih odhodnih glasovnih klicev."),
    "povp_minute_konice": ("mou_peav_Mean", "Povprečne glasovne minute v konici."),
    "povp_minute_izven_konic": ("mou_opkv_Mean", "Povprečne glasovne minute zunaj konice."),
    "povp_klici_konice": ("peak_vce_Mean", "Povprečno število glasovnih klicev v konici."),
    "povp_klici_izven_konic": ("opk_vce_Mean", "Povprečno število glasovnih klicev zunaj konice."),
}
slovar_vhodov = osnovni_slovar | dodatni_slovar
preimenovanje = {vir: ime for ime, (vir, _) in slovar_vhodov.items()}
tabela = surovi[["Customer_ID", "churn", *preimenovanje]].rename(
    columns={"Customer_ID": "id_stranke", "churn": "odhod", **preimenovanje}).copy()
tabela["odhod"] = tabela["odhod"].astype("int8")
print("Izbranih izvornih značilk:", len(slovar_vhodov))

In [ ]:
# Vnaprej določena pravila; nobeno ne računa parametrov iz vseh vrstic.
def razmerje(stevec, imenovalec):
    return stevec / imenovalec.where(imenovalec.gt(0))

def kazalnik(pogoj, znano):
    return pogoj.astype(float).where(znano)

# Slovar značilk bomo izvozili skupaj s tabelo.
opisi = {}
skupine = {}
def dodaj(ime, vrednosti, skupina, opis, vir):
    tabela[ime] = vrednosti
    skupine.setdefault(skupina, []).append(ime)
    opisi[ime] = {"opis": opis, "vir": vir, "skupina": skupina}

In [ ]:
kat_stolpci = ["regija", "zakonski_stan", "splet_na_napravi", "dualband", "obnovljena_naprava"]
num_stolpci = [c for c in slovar_vhodov if c not in kat_stolpci]
naprava = {"cena_naprave", "dni_z_napravo", "stevilo_telefonov", "stevilo_modelov_telefonov",
           "splet_na_napravi", "dualband", "obnovljena_naprava"}
demografija = {"zakonski_stan", "dohodek_razred", "stevilo_avtomobilov"}
prihodki = {"povp_mesecni_prihodek", "povp_mesecna_narocnina", "povp_prihodek_3m",
            "povp_prihodek_6m", "sprememba_prihodka"}
razmerje_narocnika = {"mesecev_narocnine", "stevilo_narocnin", "aktivne_narocnine", "regija"}
for ime, (vir, opis) in slovar_vhodov.items():
    skupina = ("naprava" if ime in naprava else "demografija" if ime in demografija else
               "prihodki" if ime in prihodki else "narocnisko_razmerje" if ime in razmerje_narocnika else "poraba_in_klici")
    dodaj(ime, tabela[ime], skupina, opis, vir)

## 3. Poenotimo manjkajoče vrednosti in preverimo domenske omejitve
`U` in `UNKW` pretvorimo samo v stolpcih, kjer pomenita neznano. Nič minut
ostane veljaven podatek. Negativna trajanja, števila in redno naročnino označimo
kot manjkajoče. Negativnih sprememb ne brišemo; negativne prihodke ohranimo,
ker lahko pomenijo dobropis. Prihodki zato niso isto kot redna naročnina.

Velikih vrednosti ne odstranjujemo samo zato, ker so redke. Tudi percentilnih
mej za omejevanje v tej fazi ne učimo.

In [ ]:
for c in kat_stolpci:
    tabela[c] = tabela[c].astype("string").str.strip().str.upper().replace("", pd.NA)
for c, oznake in {"zakonski_stan": ["U"], "dualband": ["U"], "splet_na_napravi": ["UNKW"]}.items():
    tabela[c] = tabela[c].replace(oznake, pd.NA)
for c in num_stolpci:
    tabela[c] = pd.to_numeric(tabela[c], errors="coerce").replace([np.inf, -np.inf], np.nan)

# Izjeme so spremembe in prihodki, ne števci/trajanja.
lahko_negativni = {"sprememba_minut", "sprememba_prihodka", "povp_mesecni_prihodek",
                  "povp_prihodek_3m", "povp_prihodek_6m"}
popravljeno = {}
for c in num_stolpci:
    napaka = tabela[c].lt(0) if c not in lahko_negativni else pd.Series(False, index=tabela.index)
    if c == "dohodek_razred":
        napaka = tabela[c].notna() & (~tabela[c].between(1, 9) | tabela[c].mod(1).ne(0))
    popravljeno[c] = int(napaka.sum())
    tabela[c] = tabela[c].mask(napaka)
display(pd.Series(popravljeno, name="označeno kot manjkajoče").loc[lambda x: x.gt(0)].to_frame())
display(tabela[list(slovar_vhodov)].isna().mean().mul(100).sort_values(ascending=False).head(10).rename("manjka (%)").to_frame())

## 4. Izpeljemo značilke, ki jih bomo lahko primerjali z izvirnimi
Razmerja imajo pomen samo ob znanem, pozitivnem imenovalcu. Prihodki na minuto
niso cena minute: vključujejo lahko fiksno naročnino. Kazalnik stika s podporo
ni število klicev. Meje skupin trajanja (12/24 mesecev) so določene vnaprej.

Primerjamo tudi porabo minut z dodatno značilko `log1p_povp_minute = log(1 + povp_minute)`.
Logaritem stisne velike vrednosti in ohrani ničle. Izvirno porabo ohranimo;
ali dodatna značilka pomaga modelu, bomo preverili pozneje na validaciji.


In [ ]:
razmerja = [
    ("razmerje_minut_3m_6m", "povp_minute_3m", "povp_minute_6m"),
    ("razmerje_prihodka_3m_6m", "povp_prihodek_3m", "povp_prihodek_6m"),
    ("razmerje_klicev_3m_6m", "povp_klici_3m", "povp_klici_6m"),
    ("razmerje_minut_trenutno_6m", "povp_minute", "povp_minute_6m"),
    ("prihodek_na_minuto", "povp_mesecni_prihodek", "povp_minute"),
]
for ime, stevec, imenovalec in razmerja:
    dodaj(ime, razmerje(tabela[stevec], tabela[imenovalec]), "izpeljane",
          f"Razmerje {stevec} / {imenovalec}; NaN pri nepozitivnem ali neznanem imenovalcu.", f"{stevec}, {imenovalec}")
for ime, a, b in [("razlika_minut_3m_6m", "povp_minute_3m", "povp_minute_6m"),
                  ("razlika_prihodka_3m_6m", "povp_prihodek_3m", "povp_prihodek_6m")]:
    dodaj(ime, tabela[a] - tabela[b], "izpeljane", f"Razlika {a} − {b}.", f"{a}, {b}")
dodaj("delez_minut_konice", razmerje(tabela["povp_minute_konice"],
      tabela["povp_minute_konice"] + tabela["povp_minute_izven_konic"]), "izpeljane",
      "Delež minut v konici med minutami v konici in zunaj nje.", "mou_peav_Mean, mou_opkv_Mean")
dodaj("stik_s_podporo", kazalnik(tabela["povp_klici_podpora"].gt(0), tabela["povp_klici_podpora"].notna()),
      "izpeljane", "1 = pozitiven agregat stikov s podporo; neznano ostane NaN.", "custcare_Mean")
dodaj("brez_porabe_minut", kazalnik(tabela["povp_minute"].eq(0), tabela["povp_minute"].notna()),
      "izpeljane", "1 = znana ničelna poraba minut.", "mou_Mean")
dodaj("trajanje_skupina", pd.cut(tabela["mesecev_narocnine"], [0, 12, 24, np.inf],
      labels=["DO_12", "13_DO_24", "NAD_24"], include_lowest=True).astype("string").fillna("NEZNANO"),
      "izpeljane", "Fiksne skupine trajanja naročnine; izvorne mesece ohranimo.", "months")
for izvor in ["povp_minute", "cena_naprave", "povp_klici_podpora"]:
    dodaj("log1p_" + izvor, np.log1p(tabela[izvor]), "transformirane",
          "Naravni logaritem (1 + vrednost); brez učenja parametrov.", izvor)
dodaj("predznacen_log_prihodka", np.sign(tabela["povp_mesecni_prihodek"]) * np.log1p(tabela["povp_mesecni_prihodek"].abs()),
      "transformirane", "Predznačeni logaritem ohrani tudi negativne prihodke.", "rev_Mean")

fig, osi = plt.subplots(1, 2, figsize=(11, 3.5))
tabela["povp_minute"].dropna().plot.hist(bins=45, ax=osi[0], color="#2993AD")
osi[0].set(title="Poraba: velike veljavne vrednosti ostanejo", xlabel="Minute", ylabel="Naročniki")
tabela["log1p_povp_minute"].dropna().plot.hist(bins=45, ax=osi[1], color="#205781")
osi[1].set(title="Dodatna logaritemska značilka", xlabel="log(1 + minute)", ylabel="Naročniki")
fig.tight_layout(); plt.show()

## 5. Ohranimo informacijo o manjkanju
Seznam kazalnikov določimo vnaprej; ne izbiramo jih po povezavi z odhodom.
Numeričnih lukenj še ne nadomeščamo z mediano. Kategorijo `NEZNANO` pa lahko
uporabimo že zdaj, ker je stalna oznaka in se je ne učimo iz podatkov.

Spodnja tabela pokaže znane in manjkajoče vrednosti `povp_minute`.
Kazalnik `povp_minute_manjka` je 1 pri manjkajoči vrednosti in 0 pri znani
(tudi če je poraba 0 minut). Tako informacija o manjkanju ostane na voljo
modelu tudi po poznejšem nadomeščanju vrednosti z učno mediano.


In [ ]:
spremljaj_manjkanje = ["povp_minute", "povp_minute_6m", "povp_prihodek_6m", "dni_z_napravo",
    "povp_mesecna_narocnina", "dohodek_razred", "stevilo_avtomobilov", "zakonski_stan",
    "splet_na_napravi", "regija", "razmerje_minut_3m_6m", "prihodek_na_minuto"]
for c in spremljaj_manjkanje:
    dodaj(c + "_manjka", tabela[c].isna().astype("int8"), "manjkanje",
          "1 = manjka ali je po čiščenju nedoločeno; 0 = znano.", c)
for c in kat_stolpci:
    tabela[c] = tabela[c].fillna("NEZNANO")

# Primeri znane in manjkajoče porabe: kazalnik ohrani informacijo o manjkanju.
primeri = pd.concat([
    tabela.loc[tabela["povp_minute"].notna()].head(3),
    tabela.loc[tabela["povp_minute"].isna()].head(3),
])
display(primeri[["povp_minute", "povp_minute_manjka"]])

## 6. Pripravimo nabore za kasnejše poskuse
Nabori so določeni po vsebini, ne po uspešnosti modela. Nekatere izpeljave so
odvisne od izvirnih značilk, zato več stolpcev ne pomeni nujno več informacij.
Primerjavo in izbor opravimo šele na ustrezno razdeljenih podatkih.

In [ ]:
IME, KLJUC, CILJ = "telekom", "id_stranke", "odhod"
META, DATUMI = [KLJUC], []
opisi[KLJUC] = {"opis": "ID naročnika; samo povezava, ni vhod v model.", "vir": "Customer_ID", "skupina": "metapodatki"}
opisi[CILJ] = {"opis": "Oznaka odhoda 0/1; časovni horizont ni potrjen.", "vir": "churn", "skupina": "cilj"}
vse = [c for v in skupine.values() for c in v]
nabori = {
    "osnovni": ["mesecev_narocnine", "povp_minute", "povp_mesecni_prihodek", "povp_klici_podpora", "dni_z_napravo", "regija"],
    "izvorni": list(slovar_vhodov),
    "brez_demografije": [c for c in vse if c not in skupine["demografija"] and
        c not in [d + "_manjka" for d in skupine["demografija"]]],
    "vsi": vse,
}
IZLOCENI = sorted(set(surovi.columns) - set(preimenovanje) - {"Customer_ID", "churn"})
OPOMBE = ["Ni delitve, imputacije z naučeno statistiko, kodiranja, skaliranja ali modela.",
    "NaN so namerno ohranjeni; mediane in kodiranje določimo po delitvi.",
    "Časovni horizont in časovna dostopnost agregatov v izvozu nista potrjena.",
    "Veljavnih ekstremov ne režemo; negativne prihodke ohranimo, negativno redno naročnino označimo kot manjkajočo."]
assert tabela["odhod"].isin([0, 1]).all()
assert tabela["dni_z_napravo"].dropna().ge(0).all()
assert tabela["delez_minut_konice"].dropna().between(0, 1).all()
display(pd.DataFrame({"nabor": list(nabori), "značilk": [len(v) for v in nabori.values()]}))

## 7. Preverimo in shranimo čisto tabelo
CSV vsebuje cilj in ID zaradi poznejšega povezovanja. Vhodne stolpce vedno
vzamemo iz sheme; ne uporabimo avtomatično »vseh stolpcev razen cilja«.
Ponovni zagon prepiše samo ta dva pripravljena izhoda.

In [ ]:
znacilke = [c for stolpci in skupine.values() for c in stolpci]
assert len(znacilke) == len(set(znacilke))
assert set(tabela.columns) == set(META + [CILJ] + znacilke)
assert not set(META + [CILJ]) & set(znacilke)
assert tabela[KLJUC].notna().all() and tabela[KLJUC].is_unique
assert tabela[CILJ].notna().all()
assert not np.isinf(tabela.select_dtypes(include="number").to_numpy(dtype=float)).any()
assert all(set(v) <= set(znacilke) for v in nabori.values())

kategoricne = [c for c in znacilke if not pd.api.types.is_numeric_dtype(tabela[c])]
numericne = [c for c in znacilke if c not in kategoricne]
tabela = tabela[META + [CILJ] + znacilke].sort_values(KLJUC).reset_index(drop=True)
slovar = []
for c in tabela:
    vloga = "cilj" if c == CILJ else ("metapodatek" if c in META else "znacilka")
    slovar.append({"stolpec": c, "vloga": vloga, "dtype": str(tabela[c].dtype),
        "manjkajocih": int(tabela[c].isna().sum()), **opisi[c]})
shema = {
    "razlicica": 1, "zbirka": IME, "datoteka": f"{IME}_modeliranje.csv",
    "kljuc": KLJUC, "cilj": CILJ, "metapodatki": META, "datumski_stolpci": DATUMI,
    "numericne_znacilke": numericne, "kategoricne_znacilke": kategoricne,
    "znacilke": znacilke, "skupine_znacilk": skupine, "nabori_znacilk": nabori,
    "stolpci": slovar, "vrstic": len(tabela), "stevilo_znacilk": len(znacilke),
    "izloceni_vhodi": IZLOCENI, "opombe": OPOMBE,
}
pot_csv = IZHOD / shema["datoteka"]
pot_json = IZHOD / f"{IME}_shema.json"
tabela.to_csv(pot_csv, index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
pot_json.write_text(json.dumps(shema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

# Preverimo, da CSV in shema ohranita tipe, ključe in manjkajoče vrednosti.
tipi = {r["stolpec"]: r["dtype"] for r in slovar if r["stolpec"] not in DATUMI}
prebrana = pd.read_csv(pot_csv, dtype=tipi, parse_dates=DATUMI,
    keep_default_na=False, na_values=[""])
pd.testing.assert_frame_equal(tabela, prebrana, check_exact=False, rtol=1e-10, atol=1e-10)
print(f"Shranjeno: {len(tabela):,} vrstic, {len(znacilke)} značilk + cilj in metapodatki.")
print(pot_csv.relative_to(KOREN))
print(pot_json.relative_to(KOREN))
display(pd.DataFrame([{"skupina": k, "značilk": len(v)} for k, v in skupine.items()]))
display(tabela.head(3))

## Predaja naslednjemu poglavju
Tukaj se ustavimo: dve vlogi podatkov in seznam značilk so pripravljeni, vrstice
pa še niso razdeljene. Pri naslednjem poglavju bomo najprej izbrali način delitve,
nato znotraj učnega dela naučili pripravo in model.

In [ ]:
# To je vhod za naslednji sklop. Še vedno ni delitve ali modela.
shema = json.loads(pot_json.read_text(encoding="utf-8"))
X = prebrana[shema["znacilke"]].copy()
y = prebrana[shema["cilj"]].copy()
metapodatki = prebrana[shema["metapodatki"]].copy()
print("X:", X.shape, "y:", y.shape, "metapodatki:", metapodatki.shape)